In [1]:
from langchain_classic.retrievers import EnsembleRetriever # Package name moved from langchain.retrievers to langchain_classic.retrievers
from langchain_community.retrievers import BM25Retriever # Package name moved from langchain.retrievers to langchain_community.retrievers
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

from config.config import DEVICE, DB_PATH, EMBEDDING_MODEL_NAME, EMBEDDING_KWARGS
import torch

from dotenv import load_dotenv

load_dotenv()

c:\Users\lovep\miniconda3\envs\ragApp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lovep\AppData\Local\Temp\ipykernel_9020\2900574726.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever # Package name moved from langchain.retrievers to langchain_community.retrievers


False

In [2]:
# Extra info for deleting model/pipeline objects and freeing up GPU memory
# import gc

# # Delete the objects holding references to the model/pipeline
# del llm, chat  # add any other model/pipeline variables you created

# gc.collect()          # clear Python-level references
# torch.cuda.empty_cache()      # release cached (but unused) VRAM back to the OS
# torch.cuda.ipc_collect()      # clean up any inter-process memory


In [3]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [4]:
# Convert to Document objects for LangChain
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

print("Sample Data:")
for i, chunk in enumerate(chunks, 1):
    print(f"{i}. {chunk}")

print("\n" + "="*80)

Sample Data:
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java pr

# 1. Vector Retriever (Semantic Search/Dense Retrieval)

In [5]:
print("Setting up Vector retriever with HuggingFace Embeddings and Chroma vector store...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": DEVICE},
    encode_kwargs=EMBEDDING_KWARGS,
)

vector_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

Setting up Vector retriever with HuggingFace Embeddings and Chroma vector store...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11291.33it/s]


In [6]:
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 2})

# Test semantic search with a query
test_query = "space exploration company" #works in vector search but wouldn't work with keyword search

print(f"\nTesting: '{test_query}'")
test_docs = vector_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")


Testing: 'space exploration company'


c:\Users\lovep\miniconda3\envs\ragApp\Lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Found: == Corporate affairs ==


=== List of chief executives ===
Martin Eberhard (2004–2007)
Ze'ev Drori (2007–2008)
Elon Musk (since October 2008)


=== List of board chairs ===
Elon Musk (2004–2018)
Robyn Denholm (since November 2018)
Found: Tesla has received criticism that its board lacks enough independent directors. In an April 2017 public letter, a group of influential Tesla investors, including the California State Teachers' Retirement System, asked Tesla to add two new independent directors to its board "who do not have any ties" with Musk, writing that "five of six current non-executive directors have professional or personal ties to Mr. Musk that could put at risk their ability to exercise independent judgement." Tesla's directors at the time included Brad Buss, who served as chief financial officer at SolarCity; Steve Jurvetson, a venture capitalist who also sits on the board of SpaceX; Elon Musk's brother, Kimbal; and Ira Ehrenpreis and Antonio Gracias, both of whom also 

# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [7]:
print("Setting up BM25 retriever for keyword search...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3  # Set the number of documents to retrieve

Setting up BM25 retriever for keyword search...


In [8]:
# Test exact keyword matching
# test_query = "space exploration company"
test_query = "Cybertruck"
# test_query = "Tesla"

print(f"\nKeyword Search Test Query: '{test_query}'")
test_docs = bm25_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Retrieved: {doc.page_content}")


Keyword Search Test Query: 'Cybertruck'
Retrieved: Cybertruck reservations exceeded company expectations.
Retrieved: Tesla Cybertruck production ramp begins in 2024.
Retrieved: Orange juice sales increased during winter months.


# 3. Hybrid Retriever (Combination)

In [9]:
# Hybrid search: Combine BM25 and vector search results
print("\nSetting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # Adjust weights as needed for your use case
)
print("Setup complete. You can now use 'hybrid_retriever' to perform hybrid searches combining both semantic and keyword search results.")


Setting up Hybrid Retriever...
Setup complete. You can now use 'hybrid_retriever' to perform hybrid searches combining both semantic and keyword search results.


In [10]:
# Query 1: Mixed semantic and exact terms

# Vector search understands "purchase cost" semantically
# BM25 search finds exact "7.5 billion" 
# Hybrid should combine both strengths for best result
test_query = "purchase cost 7.5 billion"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print(80*"=")

print("Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching")

1. . China's refusal to buy them could cost the company $30 billion.
2. . This would have been the largest semiconductor acquisition in history.
3. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
4. Orange County reported new housing developments.
5. Orange juice sales increased during winter months.
Query 1 shows how hybrid finds exact financial info using both semantic understanding and keyword matching


In [11]:
# Query 2: Semantic concept + specific product name  

# Vector search understands "electric vehicle manufacturing"
# BM25 search finds exact "Cybertruck"
# Hybrid gets the best of both worlds

test_query = "electric vehicle manufacturing Cybertruck"

retrieved_chunks = hybrid_retriever.invoke(test_query)

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print(80*"=")
print("Query 2 demonstrates combining product-specific terms with broader concepts")

1. Between May 2023 and February 2024, almost all major North America EV manufacturers announced plans to switch to Tesla's North American Charging Standard adapters on their EVs by 2025, which is expected to be a stable source of recurring revenue for Tesla. In November, Tesla started shipping the Cybertruck, produced from Gigafactory Texas.
In April 2024, the company announced it was laying off 10% of its employees. In June, the company moved its incorporation from Delaware to Texas. In October, the company unveiled a concept version of two autonomous vehicles – the Cybercab and Robovan – and detailed that both would be an integral part of a Tesla ridehailing service called the Tesla Network, a future service it had previously teased in 2019.
2. The Cybertruck is a full-sized pickup truck. First announced in November 2019, pilot production began in July 2023, after being pushed back multiple times, and deliveries began on November 30, 2023. Three models are offered: rear-wheel drive,

In [12]:
# Query 3: Where neither alone would be perfect

# "Company performance" is semantic, "Tesla" is exact keyword
# Hybrid should find the most relevant Tesla performance info

test_query = "company performance Tesla"

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print(80*"=")

print("Query 3 shows how hybrid handles mixed semantic/keyword queries better than either approach alone")

1. === Global expansion and Model Y (2019–present) ===
From July 2019 to June 2020, Tesla reported four consecutive profitable quarters for the first time, which made it eligible for inclusion in the S&P 500. During 2020, its share price increased 740%, and by December 14, 2020, its market capitalization was more than the next nine largest automakers combined, and it became the sixth most valuable company in the US. Tesla was added to the S&P index on December 21, 2020; it was the most valuable company ever added, and was the sixth-largest member of the index immediately after it was added. It became the sixth US company to reach $1 trillion in market capitalization in October 2021.
2. Tesla is one of the world's most valuable companies in terms of market capitalization. Starting in July 2020, it has been the world's most valuable automaker. From October 2021 to March 2022, Tesla was a US$1 trillion company, the sixth US company to reach that valuation. In 2023, the company was ranked 

In [13]:
# Combine the query and the relevant document contents
combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

# Set up the HuggingFace model for text generation
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    device_map="auto",
    model_kwargs={"dtype": torch.float16},          # load weights in fp16 to fit VRAM
    pipeline_kwargs={
        "temperature": 0.2,
        "do_sample": True,          # required for temperature to have any effect
        "max_new_tokens": 512,      # explicit cap, avoids conflicting with the model's default max_length=20
        "return_full_text": False,  # return ONLY the generated answer, not prompt+answer glued together
    },
)

chat = ChatHuggingFace(llm=llm)

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = chat.invoke(messages)

# Display the full result and content only
print("\n--- Generated Response ---")
# print("Full result:")
# print(result)
print("Content only:")
print(result.content)

Loading weights: 100%|██████████| 339/339 [00:03<00:00, 92.24it/s] 
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corru


--- Generated Response ---
Content only:
Based on the provided documents, Tesla experienced significant growth and success during the period from 2019 to at least 2025. Here are some key points:

- From July 2019 to June 2020, Tesla reported four consecutive profitable quarters, making it eligible for inclusion in the S&P 500.
- In 2020, Tesla's share price increased by 740%, and by December 14, 2020, its market capitalization surpassed the combined value of the next nine largest automakers, positioning it as the sixth most valuable company in the U.S.
- Tesla was added to the S&P 500 index on December 21, 2020, becoming the most valuable company ever added to the index.
- In October 2021, Tesla became the sixth US company to reach $1 trillion in market capitalization.
- From November 2024 to February 2025, and again from May 2025 to July 2026, Tesla's market capitalization exceeded $1 trillion.
- In November 2025, Tesla approved a $1 trillion pay package for Elon Musk, contingent on 